# 01 — Data Cleaning

**Goal:** take the messy `data/raw/` files and turn them into a clean,
consistent, analysis-ready dataset saved to `data/clean/`.

This notebook is the *first stage* of the project. In the real world this is
where most analyst time goes — you can't trust any analysis until the data
underneath it is clean. Each section below:

1. **Shows** the problem in the raw data,
2. **Explains** why it's a problem,
3. **Fixes** it, and
4. **Verifies** the fix worked.

> The data is synthetic (see `DATA_DICTIONARY.md`), but the *mess* is the kind
> you genuinely meet: mixed date formats, inconsistent labels, numbers stored
> as text, duplicates, and missing values.

---

### Cleaning checklist

| # | Issue | Tables affected |
|---|-------|-----------------|
| 1 | Duplicate rows | employees, performance_reviews |
| 2 | Whitespace & casing in text | employees, departments |
| 3 | Inconsistent categories (gender, status) | employees |
| 4 | Mixed date formats | employees, salaries |
| 5 | Numbers stored as text ("Rp..", "85%") | employees, goals |
| 6 | Missing values | several |
| 7 | Type casting & validation | all |


## Setup

We load each raw table **as strings** (`dtype=str`). This is deliberate: if we
let pandas guess types now, it would choke on values like `Rp30.100.000` or
silently turn `employee_id` into a float. We want to see the raw mess exactly as
it is, then convert types ourselves, on purpose.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/clean_rebuilt")   # we write here so we don't overwrite the provided clean/ reference
OUT.mkdir(parents=True, exist_ok=True)

# Load everything as strings first — see note above.
emp   = pd.read_csv(RAW / "employees.csv", dtype=str)
dept  = pd.read_csv(RAW / "departments.csv", dtype=str)
sal   = pd.read_csv(RAW / "salaries.csv", dtype=str)
rev   = pd.read_csv(RAW / "performance_reviews.csv", dtype=str)
goals = pd.read_csv(RAW / "goals.csv", dtype=str)

print("Raw shapes:")
for name, df in [("employees", emp), ("departments", dept),
                 ("salaries", sal), ("performance_reviews", rev), ("goals", goals)]:
    print(f"  {name:22s} {df.shape}")

Raw shapes:
  employees              (4568, 14)
  departments            (10, 4)
  salaries               (12919, 5)
  performance_reviews    (15015, 5)
  goals                  (13534, 6)


Let's actually *look* at the employees table — the messiest one — before
touching anything. Reading the raw data with your own eyes is an underrated
first step.

In [2]:
emp.head(12)

,employee_id,full_name,gender,dept_id,job_level,city,employment_type,hire_date,termination_date,status,monthly_salary_idr,satisfaction_score,last_performance_score,manager_id
0,1001,"Balidin Dongoran, S.T.",L,9,Junior,Bandung,Permanent,2015-04-13,NaN,active,16680000,4.2,3.2,5095
1,1002,Darmana Mandala,Male,7,Mid,Medan,Permanent,2022-08-03,NaN,ACTIVE,16330000,3.2,3.4,3090
2,1003,"Ciaobella Wastuti, M.Pd",P,2,Mid,Bandung,Permanent,2015-05-11,2018-04-20,Resigned,25570000,4.5,4.0,3357
3,1004,Lantar Anggraini,Male,7,Mid,Yogyakarta,Permanent,04/17/2021,2024-11-03,resigned,14590000,2.5,3.6,2039
4,1005,Among Iswahyudi,female,5,Senior,Bandung,Permanent,01/29/2024,NaN,Active,Rp30.100.000,4.9,3.8,2689
5,1006,Daruna Mayasari,NaN,2,Junior,NaN,Permanent,12-02-2018,NaN,active,20600000,4.3,4.0,2786
6,1007,CUT PUTRI NAPITUPULU,P,3,Junior,Surabaya,Permanent,2016-02-01,NaN,Active,19990000,2.6,4.1,1825
7,1008,"Sadina Palastri, S.Farm",L,7,Junior,Jakarta,Probation,2023-03-08,NaN,ACTIVE,12020000,NaN,3.7,2905
8,1009,ikin rahmawati,Female,6,Mid,Remote,Permanent,2015-11-19,NaN,ACTIVE,19610000,3.8,3.6,1724
9,1010,Yessi Ardianto,Female,8,Lead,Jakarta,Permanent,22-06-2021,NaN,active,44500000,3.2,3.9,5008


## 1. Duplicate rows

**The problem:** some rows are exact duplicates — the same record imported
twice. If we don't remove them, every count, average, and sum downstream will be
slightly wrong (e.g. an employee counted twice inflates headcount and skews
attrition).

**How we check:** `.duplicated().sum()` counts rows that are identical to an
earlier row.

In [3]:
print("Duplicate rows BEFORE:")
print("  employees          :", emp.duplicated().sum())
print("  performance_reviews:", rev.duplicated().sum())

Duplicate rows BEFORE:
  employees          : 68
  performance_reviews: 149


**The fix:** `drop_duplicates()` keeps the first occurrence and drops the
rest. We reset the index afterwards so row numbers stay tidy.

In [4]:
emp = emp.drop_duplicates().reset_index(drop=True)
rev = rev.drop_duplicates().reset_index(drop=True)

print("Duplicate rows AFTER:")
print("  employees          :", emp.duplicated().sum())
print("  performance_reviews:", rev.duplicated().sum())

Duplicate rows AFTER:
  employees          : 0
  performance_reviews: 0


## 2. Whitespace & inconsistent casing in text

**The problem:** names arrived with stray leading/trailing spaces, *double*
spaces between words, and inconsistent capitalisation (`KH.  Capa  Anggraini`,
`  some name `, `ALL CAPS`, `all lower`). Department names have trailing spaces.

Why it matters: `"Engineering"` and `"Engineering "` look identical to a human
but are **different values** to a computer — they'd break joins and split a
single category into two in any group-by.

**The fix:**
- `.str.strip()` removes leading/trailing spaces.
- A regex `\s+` → single space collapses internal double spaces.
- `.str.title()` standardises name casing (Title Case).

In [5]:
def clean_text(series):
    """Strip ends, collapse internal whitespace to single spaces."""
    return (series.astype(str)
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True))

# Names: clean whitespace, then Title Case for consistency
emp["full_name"] = clean_text(emp["full_name"]).str.title()

# Department names: just whitespace (keep their original casing)
dept["dept_name"] = clean_text(dept["dept_name"])

emp[["employee_id", "full_name"]].head(8)

,employee_id,full_name
0,1001,"Balidin Dongoran, S.T."
1,1002,Darmana Mandala
2,1003,"Ciaobella Wastuti, M.Pd"
3,1004,Lantar Anggraini
4,1005,Among Iswahyudi
5,1006,Daruna Mayasari
6,1007,Cut Putri Napitupulu
7,1008,"Sadina Palastri, S.Farm"


## 3. Inconsistent categorical labels

**The problem:** the same concept is written many ways:
- **gender**: `Male`, `male`, `M`, `L` (laki-laki) all mean the same thing;
  `Female`, `female`, `F`, `P` (perempuan) too.
- **status**: `Active`, `ACTIVE`, `active` — casing all over the place.

Left as-is, a `groupby("gender")` would show 6–8 groups instead of 2.

**The fix:** build an explicit *mapping dictionary* from every messy variant to
one canonical value. Explicit mapping is safer than clever guessing — you can
read exactly what becomes what.

In [6]:
# First, SEE every distinct value we have to handle:
print("Distinct gender values :", sorted(emp["gender"].dropna().unique()))
print("Distinct status values :", sorted(emp["status"].dropna().unique()))

Distinct gender values : ['F', 'Female', 'L', 'M', 'Male', 'P', 'female', 'male']
Distinct status values : ['ACTIVE', 'Active', 'RESIGNED', 'Resigned', 'active', 'resigned']


In [7]:
gender_map = {
    "male": "Male", "m": "Male", "l": "Male",
    "female": "Female", "f": "Female", "p": "Female",
}
def map_gender(v):
    if pd.isna(v):
        return np.nan
    return gender_map.get(str(v).strip().lower(), np.nan)

emp["gender"] = emp["gender"].apply(map_gender)

# status: simply normalise casing to Title Case
emp["status"] = emp["status"].str.strip().str.title()

print("AFTER:")
print("  gender:", emp["gender"].value_counts(dropna=False).to_dict())
print("  status:", emp["status"].value_counts(dropna=False).to_dict())

AFTER:
  gender: {'Female': 2173, 'Male': 2147, nan: 180}
  status: {'Active': 3814, 'Resigned': 686}


## 4. Mixed date formats

**The problem:** dates come in several formats in the same column:
`2021-04-17` (ISO), `17/04/2021` (DD/MM/YYYY), `04/17/2021` (MM/DD/YYYY),
`17-04-2021` (DD-MM-YYYY). If you parse them with a single fixed format you'll
either error out or — worse — silently swap day and month.

**The fix:** a small parser that tries the known formats in order and returns
the first that works. We convert to real `datetime`, then store back as clean
ISO `YYYY-MM-DD` strings (unambiguous, sorts correctly, universal).

> Note: `04/17/2021` is clearly MM/DD (no 17th month), but `05/04/2021` is
> ambiguous. Because we *generated* this data we know the non-ISO slash format
> is DD/MM; in a real project you'd confirm the source's convention before
> trusting it. We document the assumption right in the code.

In [8]:
from datetime import datetime

# Order matters: try the most specific / most likely first.
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y"]

def parse_date(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    s = str(value).strip()
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return pd.NaT   # nothing matched — flag as missing so we can inspect later

for col in ["hire_date", "termination_date"]:
    emp[col] = emp[col].apply(parse_date)

sal["effective_date"] = sal["effective_date"].apply(parse_date)

# How many failed to parse? (should be 0 for hire/effective dates)
print("Unparsed hire_date     :", emp["hire_date"].isna().sum())
print("Unparsed effective_date:", sal["effective_date"].isna().sum())
emp[["employee_id", "hire_date", "termination_date"]].head(6)

Unparsed hire_date     : 0
Unparsed effective_date: 0


,employee_id,hire_date,termination_date
0,1001,2015-04-13,NaT
1,1002,2022-08-03,NaT
2,1003,2015-05-11,2018-04-20
3,1004,2021-04-17,2024-11-03
4,1005,2024-01-29,NaT
5,1006,2018-02-12,NaT


## 5. Numbers stored as text

**The problem:**
- `monthly_salary_idr` sometimes looks like `Rp30.100.000` (Indonesian
  thousands use `.`) or has a trailing space.
- `achievement_pct` in goals sometimes looks like `"85%"`.

You can't average or sum text. We need clean numbers.

**The fix:** strip out everything that isn't a digit, then cast to a numeric
type. For salary we remove `Rp`, dots, and spaces. For achievement we drop the
`%`.

In [9]:
def clean_salary(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().replace("Rp", "").replace(".", "").replace(",", "").replace(" ", "")
    return pd.to_numeric(s, errors="coerce")

emp["monthly_salary_idr"] = emp["monthly_salary_idr"].apply(clean_salary).astype("Int64")

# achievement_pct: remove % then to numeric
goals["achievement_pct"] = (goals["achievement_pct"].astype(str)
                            .str.replace("%", "", regex=False)
                            .str.strip())
goals["achievement_pct"] = pd.to_numeric(goals["achievement_pct"], errors="coerce")

print("Salary dtype     :", emp["monthly_salary_idr"].dtype)
print("Salary sample    :", emp["monthly_salary_idr"].head(5).tolist())
print("Achievement range:", goals["achievement_pct"].min(), "to", goals["achievement_pct"].max())

Salary dtype     : Int64
Salary sample    : [16680000, 16330000, 25570000, 14590000, 30100000]
Achievement range: 30.0 to 130.0


## 6. Missing values

**The problem:** several columns have gaps. But **not every blank is wrong** —
context decides what to do with each:

| Column | Missing means | What we do |
|--------|---------------|-----------|
| `termination_date` | Person hasn't left | Leave blank — it's *correct* |
| `gender`, `city`, `employment_type` | Truly unknown | Fill with `"Unknown"` so it groups cleanly |
| `satisfaction_score` | Survey not completed | Leave as NaN (don't invent a score) |
| `change_reason` (salaries) | Reason not recorded | Fill with `"Unknown"` |
| `weight_pct` (goals) | Weight not set | Leave as NaN (don't guess a weight) |

The key lesson: **never blindly fill all missing values.** Filling a missing
satisfaction score with, say, the average would fabricate data and bias the
analysis. We only fill where a sensible default genuinely exists.

In [10]:
# Categorical "unknown is a category" columns -> fill with "Unknown"
for col in ["gender", "city", "employment_type"]:
    emp[col] = emp[col].fillna("Unknown")

sal["change_reason"] = sal["change_reason"].fillna("Unknown")

# Numeric/measured values we must NOT invent -> leave as NaN on purpose
# (satisfaction_score, weight_pct stay missing)

print("Remaining missing in employees:")
print(emp.isna().sum()[emp.isna().sum() > 0].to_string())
print("\n(termination_date missing = active employees; satisfaction = surveys not returned — both expected)")

Remaining missing in employees:
termination_date      3814
satisfaction_score     360
manager_id              44

(termination_date missing = active employees; satisfaction = surveys not returned — both expected)


## 7. Final type casting & validation

Now that values are clean, we lock in the correct data types and run a few
**sanity checks** — cheap assertions that would catch a mistake before it
reaches the analysis.

In [11]:
# Cast ID and numeric columns to proper types
emp["employee_id"] = pd.to_numeric(emp["employee_id"], errors="coerce").astype("Int64")
emp["dept_id"]     = pd.to_numeric(emp["dept_id"], errors="coerce").astype("Int64")
emp["manager_id"]  = pd.to_numeric(emp["manager_id"], errors="coerce").astype("Int64")
emp["satisfaction_score"] = pd.to_numeric(emp["satisfaction_score"], errors="coerce")
emp["last_performance_score"] = pd.to_numeric(emp["last_performance_score"], errors="coerce")

dept["dept_id"] = pd.to_numeric(dept["dept_id"], errors="coerce").astype("Int64")
dept["salary_band_mid"] = pd.to_numeric(dept["salary_band_mid"], errors="coerce")

for c in ["salary_id", "employee_id"]:
    sal[c] = pd.to_numeric(sal[c], errors="coerce").astype("Int64")
sal["monthly_salary_idr"] = pd.to_numeric(sal["monthly_salary_idr"], errors="coerce").astype("Int64")

for c in ["review_id", "employee_id"]:
    rev[c] = pd.to_numeric(rev[c], errors="coerce").astype("Int64")
rev["review_score"] = pd.to_numeric(rev["review_score"], errors="coerce")
rev["reviewer_id"] = pd.to_numeric(rev["reviewer_id"], errors="coerce").astype("Int64")

for c in ["goal_id", "employee_id"]:
    goals[c] = pd.to_numeric(goals[c], errors="coerce").astype("Int64")
goals["weight_pct"] = pd.to_numeric(goals["weight_pct"], errors="coerce")

print("Types locked in.")

Types locked in.


In [12]:
# --- Validation checks (assertions fail loudly if something is wrong) ---

# a) employee_id is unique
assert emp["employee_id"].is_unique, "employee_id should be unique!"

# b) every dept_id in employees exists in departments (referential integrity)
orphans = set(emp["dept_id"].dropna()) - set(dept["dept_id"].dropna())
assert not orphans, f"employees reference missing departments: {orphans}"

# c) every manager_id (if present) is a real employee
mgr = set(emp["manager_id"].dropna())
ids = set(emp["employee_id"].dropna())
assert mgr.issubset(ids), "some manager_id values aren't real employees!"

# d) gender only has expected values
assert set(emp["gender"].unique()).issubset({"Male", "Female", "Unknown"})

# e) salary is positive where present
assert (emp["monthly_salary_idr"].dropna() > 0).all(), "found non-positive salary"

print("All validation checks passed ✅")

All validation checks passed ✅


## 8. Save the cleaned data

We write the cleaned tables to `data/clean_rebuilt/`. (The project also ships a
reference `data/clean/` — keeping them separate lets you diff your result
against the target.)

In [13]:
emp.to_csv(OUT / "employees.csv", index=False)
dept.to_csv(OUT / "departments.csv", index=False)
sal.to_csv(OUT / "salaries.csv", index=False)
rev.to_csv(OUT / "performance_reviews.csv", index=False)
goals.to_csv(OUT / "goals.csv", index=False)

print("Cleaned files written to", OUT.resolve())
for f in sorted(OUT.glob("*.csv")):
    print("  ", f.name)

Cleaned files written to /home/claude/hr-analytics/data/clean_rebuilt
   departments.csv
   employees.csv
   goals.csv
   performance_reviews.csv
   salaries.csv


## Summary — what we did

Starting from raw, messy exports we:

1. Removed **duplicate rows** that would have inflated counts.
2. Stripped whitespace and standardised **text casing**.
3. Mapped inconsistent **gender/status labels** to canonical values.
4. Parsed **mixed date formats** into clean ISO dates.
5. Converted **text-encoded numbers** (`Rp…`, `85%`) into real numerics.
6. Handled **missing values** thoughtfully — filling only where a sensible
   default exists, never inventing measured values.
7. Locked in **data types** and ran **validation checks** for integrity.

The result is a trustworthy dataset ready for SQL modelling (stage 2) and
analysis (stage 3).

**Next:** `sql/` — joining these tables to answer real questions, and
`02_analysis.ipynb` — exploring attrition, performance, and pay.